# Model comparison table

`build_model_comparison_table(run_id)` builds one row per model under `results/<run_id>/` with:

- Structure adherence (`valid_structure_count/bt_count`) summed over all tasks
- Structure adherence % (2 dp)
- Task completion (`tasks_completed/total_tasks`)
- Task completion % (2 dp)
- Avg inference time (seconds)

In [7]:
import sys
import json
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.utils import get_results_dir


def build_model_comparison_table(run_id: str) -> pd.DataFrame:
    results_dir = get_results_dir(run_id)
    rows = []
    for model_dir in sorted(p for p in results_dir.iterdir() if p.is_dir()):
        main_path = model_dir / "main_results.json"
        if not main_path.exists():
            continue

        main = json.loads(main_path.read_text(encoding="utf-8"))
        all_tasks = main.get("all_tasks", [])
        if not all_tasks:
            continue

        valid_structure_count = sum(int(task.get("valid_structure_count", 0)) for task in all_tasks)
        bt_count = sum(int(task.get("bt_count", 0)) for task in all_tasks)
        adherence_ratio = f"{valid_structure_count}/{bt_count}" if bt_count else "0/0"

        tasks_completed = sum(1 for task in all_tasks if bool(task.get("task_completion", False)))
        total_tasks = len(all_tasks)

        # Fall back to top-level totals when present to stay consistent with summary fields.
        tasks_completed = int(main.get("total_task_completion", tasks_completed))
        total_tasks = int(main.get("total_tasks", total_tasks))
        task_completion_ratio = f"{tasks_completed}/{total_tasks}" if total_tasks else "0/0"

        rows.append({
            "model_id": main.get("model_id", model_dir.name),
            "Structure adherence": adherence_ratio,
            "Structure adherence %": f"{float(main.get('structure_adherence_pct', 0.0)):.2f}%",
            "Task completion": task_completion_ratio,
            "Task completion %": f"{float(main.get('task_completion_pct', 0.0)):.2f}%",
            "avg_inference_time": round(float(main.get("avg_inference_time_s", 0.0)), 3),
        })

    return pd.DataFrame(rows).sort_values("model_id").reset_index(drop=True)


In [8]:
build_model_comparison_table("bt_online_all_tasks_h100")

,model_id,Structure adherence,Structure adherence %,Task completion,Task completion %,avg_inference_time
0,Qwen/Qwen2.5-14B-Instruct,264/272,97.06%,18/100,18.00%,7.723
1,Qwen/Qwen3-14B,291/291,100.00%,7/100,7.00%,4.024
